# Notebook de Test - Pipeline d'Ingestion SFTP

Ce notebook permet de tester le pipeline d'ingestion des données via SFTP.
Il utilise le flow Prefect `sftp_ingestion_pipeline` pour :
1. Se connecter au serveur SFTP
2. Lister les fichiers disponibles
3. Télécharger et parser les fichiers
4. Faire correspondre les valeurs réelles avec les prédictions
5. Mettre à jour la base de données
6. Archiver les fichiers traités

## 0. Initialisation et Configuration

In [ ]:
import os
import logging
import warnings
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# Charger les variables d'environnement
load_dotenv(find_dotenv(".env"), override=True)
load_dotenv(find_dotenv(".env.secrets"), override=True)

# Configurer le logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Ignorer certains warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

print("✅ Initialisation terminée")

## 1. Configuration SFTP

In [ ]:
from ml.config import load_config

# Charger la configuration
config = load_config(config_name="consumption")

# Récupérer la configuration SFTP
sftp_config = config.get('sftp', {})

print('=== CONFIGURATION SFTP ===')
print(f"Enabled: {sftp_config.get('enabled', False)}")
print(f"Host: {sftp_config.get('host')}")
print(f"Username: {sftp_config.get('username')}")
print(f"Port: {sftp_config.get('port', 22)}")
print(f"Remote directory: {sftp_config.get('remote_directory')}")
print(f"Archive directory: {sftp_config.get('archive_directory')}")
print(f"File pattern: {sftp_config.get('file_pattern')}")
print(f"PPK key path: {sftp_config.get('ppk_key_path')}")

## 2. Vérification de la Configuration

In [ ]:
# Vérifier que la configuration SFTP est présente et activée
if not sftp_config:
    print("❌ Configuration SFTP manquante dans consumption.yaml")
    print("→ Ajoutez la section 'sftp:' dans consumption.yaml")
elif not sftp_config.get('enabled', False):
    print("⚠️ L'intégration SFTP est désactivée")
    print("→ Set 'sftp.enabled: true' dans consumption.yaml pour activer")
else:
    print("✅ Configuration SFTP présente et activée")
    
    # Vérifier les champs requis
    required_fields = ['host', 'username', 'ppk_key_path', 'remote_directory']
    missing = [field for field in required_fields if not sftp_config.get(field)]
    
    if missing:
        print(f"⚠️ Champs requis manquants: {missing}")
    else:
        print("✅ Tous les champs requis sont présents")

## 3. Test de Connexion SFTP

In [ ]:
from ml.connectors.sftp.sftp_connector import SFTPConnector

if sftp_config.get('enabled', False):
    try:
        connector = SFTPConnector(
            host=sftp_config['host'],
            username=sftp_config['username'],
            ppk_key_path=sftp_config['ppk_key_path'],
            passphrase=sftp_config.get('passphrase'),
            port=sftp_config.get('port', 22),
            timeout=sftp_config.get('timeout', 30)
        )
        
        with connector:
            print("✅ Connexion SFTP réussie")
            
    except Exception as e:
        print(f"❌ Erreur de connexion SFTP: {e}")
        print("→ Vérifiez les paramètres de connexion et la clé PPK")
else:
    print("⚠️ SFTP désactivé, test de connexion ignoré")

## 4. Listing des Fichiers Disponibles

In [ ]:
if sftp_config.get('enabled', False):
    try:
        from ml.connectors.sftp.sftp_tasks import list_sftp_files_task
        
        files = list_sftp_files_task(
            sftp_host=sftp_config['host'],
            sftp_username=sftp_config['username'],
            ppk_key_path=sftp_config['ppk_key_path'],
            remote_directory=sftp_config['remote_directory'],
            passphrase=sftp_config.get('passphrase'),
            sftp_port=sftp_config.get('port', 22),
            sftp_timeout=sftp_config.get('timeout', 30),
            file_pattern=sftp_config.get('file_pattern', '*.csv'),
            recursive=False
        )
        
        print(f"📁 {len(files)} fichiers trouvés")
        
        if files:
            import pandas as pd
            files_df = pd.DataFrame(files)
            display(files_df[['name', 'size', 'path']])
        else:
            print("⚠️ Aucun fichier trouvé")
            
    except Exception as e:
        print(f"❌ Erreur lors du listing: {e}")
else:
    print("⚠️ SFTP désactivé, listing ignoré")

## 5. Exécution du Pipeline Complet

In [ ]:
if sftp_config.get('enabled', False):
    try:
        from ml.workflows.sftp_ingestion_flow import sftp_ingestion_pipeline
        
        # Récupérer l'URI de la base de données
        db_uri = config.get('database', {}).get('uri')
        
        if not db_uri:
            print("❌ URI de base de données manquante dans la configuration")
        else:
            print("🚀 Exécution du pipeline d'ingestion SFTP...")
            
            result = sftp_ingestion_pipeline(
                sftp_host=sftp_config['host'],
                sftp_username=sftp_config['username'],
                ppk_key_path=sftp_config['ppk_key_path'],
                db_uri=db_uri,
                remote_directory=sftp_config['remote_directory'],
                archive_directory=sftp_config.get('archive_directory', '/archived'),
                passphrase=sftp_config.get('passphrase'),
                sftp_port=sftp_config.get('port', 22),
                sftp_timeout=sftp_config.get('timeout', 30),
                file_pattern=sftp_config.get('file_pattern', '*.csv'),
                temp_local_dir=sftp_config.get('temp_local_dir', '/tmp/sftp_temp')
            )
            
            print("\n✅ Pipeline terminé")
            print(f"Status: {result.get('status')}")
            print(f"Fichiers traités: {result.get('files_count', 0)}")
            
            if result.get('summary'):
                summary = result['summary']
                print(f"\n📊 Résumé:")
                print(f"  Total fichiers: {summary.get('total_files', 0)}")
                print(f"  Réussis: {summary.get('successful', 0)}")
                print(f"  Échoués: {summary.get('failed', 0)}")
                print(f"  Enregistrements traités: {summary.get('total_records_processed', 0)}")
                print(f"  Prédictions mises à jour: {summary.get('total_predictions_updated', 0)}")
                print(f"  Taux de succès: {summary.get('success_rate', 0):.2%}")
            
    except Exception as e:
        print(f"❌ Erreur lors de l'exécution du pipeline: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️ SFTP désactivé, pipeline ignoré")

## 6. Vérification des Mises à Jour dans la Base de Données

In [ ]:
if sftp_config.get('enabled', False):
    try:
        from ml.utils.pipelines.database_handler import DatabaseHandler
        
        db_uri = config.get('database', {}).get('uri')
        db_handler = DatabaseHandler(db_uri)
        
        # Récupérer les statistiques
        stats = db_handler.get_prediction_stats()
        
        if stats:
            print("📊 Statistiques de la base de données:")
            print(f"  Total prédictions: {stats.get('total_predictions', 0)}")
            print(f"  Table existe: {stats.get('table_exists', False)}")
        
        # Récupérer les prédictions avec valeurs réelles
        from datetime import datetime, timedelta
        today = datetime.now().date()
        yesterday = today - timedelta(days=1)
        
        predictions_with_actuals = db_handler.get_predictions_by_date(
            start_date=yesterday,
            end_date=today
        )
        
        if predictions_with_actuals is not None and not predictions_with_actuals.empty:
            with_actuals = predictions_with_actuals[predictions_with_actuals['actual_value'].notna()]
            print(f"\nPrédictions avec valeurs réelles (hier): {len(with_actuals)}")
            
            if len(with_actuals) > 0:
                display(with_actuals[['prediction_date', 'prediction', 'actual_value']].head(10))
        else:
            print("⚠️ Aucune prédiction trouvée pour la période")
            
    except Exception as e:
        print(f"❌ Erreur lors de la vérification: {e}")
else:
    print("⚠️ SFTP désactivé, vérification ignorée")

## Résumé

Ce notebook a démontré :
1. ✅ Configuration SFTP depuis consumption.yaml
2. ✅ Test de connexion SFTP
3. ✅ Listing des fichiers disponibles
4. ✅ Exécution du pipeline d'ingestion complet
5. ✅ Vérification des mises à jour dans la base de données

## Prochaines étapes
- Configurer les paramètres SFTP réels dans consumption.yaml
- Activer l'intégration SFTP avec `sftp.enabled: true`
- Configurer le scheduling automatique via Prefect
- Ajouter des alertes email en cas d'échec